# Assignment 2 — The Dirty Ledger\n\nRun this notebook in the same folder as `teaching_prices_messy.csv`. It preserves the raw data and makes uncertain records visible rather than guessing corrections.

In [ ]:
import pandas as pd\nimport numpy as np\nfrom pathlib import Path\n\npath = Path('teaching_prices_messy.csv')\nraw = pd.read_csv(path, dtype={'price': 'string'})\ndf = raw.copy()\nprint('Shape:', df.shape)\ndisplay(df.head())\nprint(df.dtypes)

## 1. Initial diagnosis\nExpected panel size = 12 dates × 4 items × 2 cities = 96 unique date-item-city combinations.

In [ ]:
print('Rows:', len(df))\nprint('Columns:', df.columns.tolist())\nprint('Missing values by column:\\n', df.isna().sum())\nprint('Exact duplicate rows:', df.duplicated().sum())\nprint('Duplicate date-item-city keys:', df.duplicated(['date','item','city'], keep=False).sum())\nprint('Dates:', df['date'].nunique(), 'Items:', df['item'].nunique(), 'Cities:', df['city'].nunique())\nprint('Item spellings:', sorted(df['item'].dropna().unique()))\nprint('Units:', df['unit'].value_counts(dropna=False))\nprint('Data labels:', df['data_label'].value_counts(dropna=False))

## 2. Cleaning with an audit trail\nThe original price is preserved in `price_raw`. Commas are removed only to parse numeric text. Suspicious values are flagged, not silently corrected.

In [ ]:
df['price_raw'] = df['price'].copy()\ndf['date'] = pd.to_datetime(df['date'], errors='coerce')\ndf['item_clean'] = df['item'].astype('string').str.strip().str.replace(r'\\s+', ' ', regex=True).str.title()\ndf['price_num'] = pd.to_numeric(df['price_raw'].str.replace(',', '', regex=False), errors='coerce')\ndf['exact_duplicate'] = df.duplicated(keep='first')\ndf['missing_price'] = df['price_num'].isna()\ndf['extreme_9999'] = df['price_num'].eq(9999)\ndf['suspicious_oil_city1_low'] = (df['item_clean'].eq('Oil B') & df['city'].eq('City 1') & df['price_num'].lt(200))\ndf['comma_formatted'] = df['price_raw'].str.contains(',', na=False)\ndf['review_flag'] = df[['missing_price','extreme_9999','suspicious_oil_city1_low']].any(axis=1)\n\n# Keep every original row in the audit table. Remove only exact repeated rows from the analysis table.\naudit = df.copy()\nclean = df.loc[~df['exact_duplicate']].copy()\ndisplay(audit.loc[audit['review_flag'] | audit['comma_formatted'] | audit['exact_duplicate'], ['date','item','item_clean','city','price_raw','price_num','exact_duplicate','review_flag','comma_formatted']])\nprint('Audit rows:', len(audit), 'Analytical rows:', len(clean))

## 3. Verification\nCheck that the key is unique after duplicate handling, that the comma-formatted value parsed as 1250, and that missing/suspicious observations remain flagged.

In [ ]:
key_dups = clean.duplicated(['date','item_clean','city'], keep=False)\nprint('Rows involved in normalized-key collisions:', key_dups.sum())\ndisplay(clean.loc[key_dups, ['date','item','item_clean','city','price_raw','price_num']])\nprint('Missing numeric prices:', clean['price_num'].isna().sum())\nprint('Comma-formatted numeric values:', clean.loc[clean['comma_formatted'], 'price_num'].tolist())\nprint('9999 flags:', clean['extreme_9999'].sum())\nprint('Low Oil B / City 1 flags:', clean['suspicious_oil_city1_low'].sum())\nprint('Rows flagged for review:', clean['review_flag'].sum())\nprint('Date parse failures:', clean['date'].isna().sum())\nprint('Unique cleaned items:', sorted(clean['item_clean'].dropna().unique()))\n\n# Expected checks for this supplied file\nassert len(raw) == 97, 'Unexpected raw row count: inspect the input file.'\nassert len(clean) == 96, 'Expected 96 rows after removing one exact duplicate.'\nassert clean['price_num'].isna().sum() == 1\nassert clean.loc[clean['comma_formatted'], 'price_num'].eq(1250).all()\nassert clean['extreme_9999'].sum() == 1\nassert clean['suspicious_oil_city1_low'].sum() == 1\nassert clean.duplicated(['date','item_clean','city']).sum() == 1, 'Expected one conflicting normalized key caused by the lowercase oil b row.'\nprint('All verification assertions passed. Note: one normalized key collision remains flagged for investigation.')

## 4. Export cleaned data and audit log\n`price_num` is the parsed numeric field. Do not treat flagged values as validated.

In [ ]:
clean.to_csv('teaching_prices_cleaned_with_flags.csv', index=False)\naudit.to_csv('teaching_prices_audit_all_rows.csv', index=False)\nprint('Saved cleaned and audit CSV files.')